# Streamlit

In [1]:
import streamlit as st
import pandas as pd
import json

# Configuración de página
st.set_page_config(page_title="Sycophancy Dataset Explorer", layout="wide")

@st.cache_data
def load_data(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

# --- Carga de datos ---
# Asegúrate de que el archivo se llame dataset.jsonl o cambia el nombre aquí
try:
    df = load_data('dataset.jsonl')
except FileNotFoundError:
    st.error("Archivo 'dataset.jsonl' no encontrado. Por favor, cárgalo en la raíz.")
    st.stop()

# --- Sidebar: Filtros Globales ---
st.sidebar.header("🎯 Filtros Globales")
all_biases = df['bias_type'].unique().tolist()
selected_biases = st.sidebar.multiselect("Filtrar por tipo de Sesgo:", all_biases, default=all_biases)

# --- Header ---
st.title("🧪 Sycophancy Multi-Bias Explorer")
st.markdown(f"Explorando **{len(df)}** interacciones distribuidas en **{df['group'].nunique()}** escenarios base.")

# --- Layout: Columnas de navegación ---
col_nav, col_view = st.columns([1, 2])

with col_nav:
    st.subheader("📁 Grupos / Escenarios")
    # Creamos una lista amigable: "Grupo X: Situación Básica"
    group_options = df.groupby('group')['basic_situation'].first().reset_index()
    group_list = [f"Grupo {row['group']}: {row['basic_situation']}" for _, row in group_options.iterrows()]
    
    selected_group_str = st.radio("Selecciona un escenario para inspeccionar:", group_list)
    selected_group_id = int(selected_group_str.split(":")[0].split(" ")[1])

# --- Filtrado de datos según selección ---
group_df = df[df['group'] == selected_group_id]
control_item = group_df[group_df['bias_type'].isna()].iloc[0]
induced_items = group_df[group_df['bias_type'].isin(selected_biases) & df['bias_type'].notna()]

with col_view:
    st.subheader("🔍 Detalle del Escenario")
    
    # 1. Mostrar el CONTROL
    st.info(f"**PROMPT DE CONTROL (Original)**\n\n{control_item['dilemma_situation']}")
    
    st.divider()
    
    # 2. Comparador de variaciones
    st.write(f"### Variaciones Inducidas ({len(induced_items)})")
    
    for _, item in induced_items.iterrows():
        # Usamos un expander para no saturar la pantalla
        label = f"ID {item['idx']} | Bias: {item['bias_type']} | Outcome: {item['outcome']}"
        with st.expander(label):
            # Resaltamos la diferencia (esto es visualmente muy potente)
            original = control_item['dilemma_situation']
            induced = item['dilemma_situation']
            
            # Encontramos dónde empieza la "inyección" del sesgo
            # (Asumiendo que el sesgo se añade al final del prompt de control)
            if induced.startswith(original):
                bias_part = induced[len(original):]
                st.write("**Texto Base:**")
                st.text(original)
                st.write("**Inyección de Sesgo:**")
                st.success(bias_part)
            else:
                st.write("**Prompt Completo:**")
                st.write(induced)

# --- Tabla Maestra (Opcional, abajo) ---
st.divider()
st.subheader("📊 Vista de Datos Cruda")
st.dataframe(df[df['bias_type'].isin(selected_biases)], use_container_width=True)

ModuleNotFoundError: No module named 'streamlit'